<a href="https://colab.research.google.com/github/aranavkumar/Veloura/blob/main/12_Clinical_Validation_Dataset_Readiness.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

# ============================================================================
# VELoura — PHASE 12, CELL 1
# DATASET, IMAGE AND ANNOTATION INVENTORY
# ============================================================================

from google.colab import drive
from pathlib import Path
from collections import Counter
import pandas as pd
import json
import re
from datetime import datetime, timezone

print("=" * 80)
print("VELoura — PHASE 12: CLINICAL VALIDATION & DATASET READINESS")
print("CELL 1: DATASET, IMAGE AND ANNOTATION INVENTORY")
print("=" * 80)

# ---------------------------------------------------------------------------
# 1. Mount Google Drive
# ---------------------------------------------------------------------------

drive.mount("/content/drive_phase12", force_remount=False)

PROJECT_DIR = Path(
    "/content/drive_phase12/MyDrive/Veloura_Project"
)

if not PROJECT_DIR.exists():
    raise FileNotFoundError(
        f"Veloura project directory not found: {PROJECT_DIR}"
    )

DATASET_DIR = PROJECT_DIR / "datasets"

PHASE12_OUTPUT = PROJECT_DIR / "outputs" / "clinical_validation"
PHASE12_REPORTS = PROJECT_DIR / "reports" / "clinical_validation"

PHASE12_OUTPUT.mkdir(parents=True, exist_ok=True)
PHASE12_REPORTS.mkdir(parents=True, exist_ok=True)

print("\nProject directory:", PROJECT_DIR)
print("Dataset directory:", DATASET_DIR)

if not DATASET_DIR.exists():
    raise FileNotFoundError(
        "Dataset directory does not exist. Check the project structure."
    )

# ---------------------------------------------------------------------------
# 2. Inventory dataset files
# ---------------------------------------------------------------------------

IMAGE_EXTENSIONS = {
    ".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"
}

MASK_EXTENSIONS = {
    ".png", ".bmp", ".tif", ".tiff"
}

ANNOTATION_KEYWORDS = [
    "mask", "masks", "annotation", "annotations",
    "ground_truth", "groundtruth", "segmentation",
    "label", "labels", "expert", "contour",
    "boundary", "reference"
]

DATASET_FILE_RECORDS = []
image_records = []
metadata_records = []
annotation_candidates = []

for path in DATASET_DIR.rglob("*"):

    if not path.is_file():
        continue

    suffix = path.suffix.lower()
    relative_path = str(path.relative_to(PROJECT_DIR))

    record = {
        "relative_path": relative_path,
        "filename": path.name,
        "extension": suffix,
        "size_bytes": path.stat().st_size
    }

    DATASET_FILE_RECORDS.append(record)

    if suffix in IMAGE_EXTENSIONS:
        image_records.append(record)

    if suffix in {".csv", ".tsv", ".json", ".jsonl"}:
        metadata_records.append(record)

    name_lower = path.name.lower()
    path_lower = str(path).lower()

    if any(
        keyword in name_lower or keyword in path_lower
        for keyword in ANNOTATION_KEYWORDS
    ):
        annotation_candidates.append(record)

inventory_df = pd.DataFrame(DATASET_FILE_RECORDS)
images_df = pd.DataFrame(image_records)
metadata_files_df = pd.DataFrame(metadata_records)
annotation_candidates_df = pd.DataFrame(annotation_candidates)

print("\nDATASET INVENTORY")
print("-" * 80)
print("Total dataset files:", len(DATASET_FILE_RECORDS))
print("Image files:", len(image_records))
print("Metadata files:", len(metadata_records))
print("Potential annotation-related files:", len(annotation_candidates))

# ---------------------------------------------------------------------------
# 3. Image file distribution
# ---------------------------------------------------------------------------

image_extensions = Counter(
    Path(x["filename"]).suffix.lower()
    for x in image_records
)

extension_df = pd.DataFrame(
    [
        {"extension": ext, "file_count": count}
        for ext, count in sorted(image_extensions.items())
    ]
)

print("\nIMAGE FILE DISTRIBUTION")
display(extension_df)

# ---------------------------------------------------------------------------
# 4. Inspect metadata tables without changing source files
# ---------------------------------------------------------------------------

metadata_schema_records = []

for item in metadata_records:

    path = PROJECT_DIR / item["relative_path"]

    try:
        if path.suffix.lower() == ".csv":
            df = pd.read_csv(path, nrows=5)
        elif path.suffix.lower() == ".tsv":
            df = pd.read_csv(path, sep="\t", nrows=5)
        else:
            metadata_schema_records.append({
                "file": item["relative_path"],
                "status": "PRESENT_NOT_TABULARLY_INSPECTED",
                "columns": None,
                "sample_rows": None
            })
            continue

        metadata_schema_records.append({
            "file": item["relative_path"],
            "status": "READ_SUCCESSFULLY",
            "columns": list(df.columns),
            "sample_rows": len(df)
        })

    except Exception as e:
        metadata_schema_records.append({
            "file": item["relative_path"],
            "status": "READ_ERROR",
            "columns": None,
            "sample_rows": str(e)[:300]
        })

metadata_schema_df = pd.DataFrame(metadata_schema_records)

print("\nMETADATA FILE SCHEMA")
display(metadata_schema_df)

# ---------------------------------------------------------------------------
# 5. Identify potential expert annotation assets
# ---------------------------------------------------------------------------

annotation_records = []

for item in annotation_candidates:

    path = PROJECT_DIR / item["relative_path"]

    annotation_records.append({
        "relative_path": item["relative_path"],
        "filename": item["filename"],
        "extension": item["extension"],
        "size_bytes": item["size_bytes"],
        "possible_mask_file": path.suffix.lower() in MASK_EXTENSIONS,
        "filename_or_path_indicates_annotation": True,
        "expert_annotation_confirmed": False
    })

annotation_inventory_df = pd.DataFrame(annotation_records)

print("\nANNOTATION DISCOVERY")
if len(annotation_inventory_df):
    display(annotation_inventory_df)
else:
    print("No annotation-related filenames discovered.")

print(
    "\nIMPORTANT: A filename containing 'mask' or 'annotation' "
    "does not prove that the file is expert-annotated."
)

# ---------------------------------------------------------------------------
# 6. Identify existing Phase 10 experimental masks separately
# ---------------------------------------------------------------------------

EXPERIMENTAL_MASK_DIR = (
    PROJECT_DIR
    / "outputs"
    / "explainability_evidence"
    / "experimental_masks"
)

experimental_mask_files = []

if EXPERIMENTAL_MASK_DIR.exists():

    for path in EXPERIMENTAL_MASK_DIR.rglob("*"):

        if path.is_file() and path.suffix.lower() in MASK_EXTENSIONS:
            experimental_mask_files.append({
                "relative_path": str(path.relative_to(PROJECT_DIR)),
                "filename": path.name,
                "size_bytes": path.stat().st_size,
                "evidence_type": "EXPERIMENTAL_VELOURA_GENERATED_MASK",
                "expert_reference": False
            })

experimental_masks_df = pd.DataFrame(experimental_mask_files)

print("\nEXPERIMENTAL MASK INVENTORY")
print("Experimental mask files:", len(experimental_mask_files))

if len(experimental_masks_df):
    display(experimental_masks_df.head(20))

# ---------------------------------------------------------------------------
# 7. Identify known six-case reference image IDs
# ---------------------------------------------------------------------------

REFERENCE_IDS = [
    "ISIC_0025481",
    "ISIC_0028145",
    "ISIC_0031157",
    "ISIC_0031437",
    "ISIC_0033038",
    "ISIC_0034119"
]

image_id_pattern = re.compile(r"ISIC_\d+")

available_image_ids = set()

for item in image_records:

    matches = image_id_pattern.findall(item["filename"])

    for image_id in matches:
        available_image_ids.add(image_id)

reference_image_audit = []

for image_id in REFERENCE_IDS:

    matches = [
        item for item in image_records
        if image_id in item["filename"]
    ]

    reference_image_audit.append({
        "image_id": image_id,
        "image_file_found": len(matches) > 0,
        "matching_image_file_count": len(matches),
        "image_paths": [
            item["relative_path"] for item in matches
        ]
    })

reference_image_audit_df = pd.DataFrame(reference_image_audit)

print("\nPHASE 5–11 REFERENCE IMAGE AUDIT")
display(reference_image_audit_df)

# ---------------------------------------------------------------------------
# 8. Save Phase 12 inventory reports
# ---------------------------------------------------------------------------

inventory_path = (
    PHASE12_REPORTS / "phase12_dataset_file_inventory.csv"
)

metadata_path = (
    PHASE12_REPORTS / "phase12_metadata_schema_inventory.json"
)

annotation_path = (
    PHASE12_REPORTS / "phase12_annotation_discovery.csv"
)

experimental_path = (
    PHASE12_REPORTS / "phase12_experimental_mask_inventory.csv"
)

reference_path = (
    PHASE12_REPORTS / "phase12_reference_image_audit.csv"
)

inventory_df.to_csv(inventory_path, index=False)
annotation_inventory_df.to_csv(annotation_path, index=False)
experimental_masks_df.to_csv(experimental_path, index=False)
reference_image_audit_df.to_csv(reference_path, index=False)

with open(metadata_path, "w", encoding="utf-8") as f:
    json.dump(
        metadata_schema_records,
        f,
        indent=2,
        ensure_ascii=False
    )

# ---------------------------------------------------------------------------
# 9. Generate initial dataset readiness summary
# ---------------------------------------------------------------------------

summary = {
    "project": "Veloura Agentic AI 2026",
    "phase": 12,
    "cell": 1,
    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "dataset_directory": str(DATASET_DIR),
    "total_dataset_files": len(DATASET_FILE_RECORDS),
    "image_file_count": len(image_records),
    "metadata_file_count": len(metadata_records),
    "annotation_candidate_count": len(annotation_candidates),
    "experimental_mask_file_count": len(experimental_mask_files),
    "reference_case_count": len(REFERENCE_IDS),
    "reference_images_found": int(
        reference_image_audit_df["image_file_found"].sum()
    ),
    "expert_annotation_status": "NOT_YET_CONFIRMED",
    "clinical_validation_status": "NOT_CLINICALLY_VALIDATED",
    "source_dataset_modified": False
}

summary_path = (
    PHASE12_OUTPUT / "phase12_dataset_inventory_summary.json"
)

with open(summary_path, "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)

# ---------------------------------------------------------------------------
# 10. Final output
# ---------------------------------------------------------------------------

print("\n" + "=" * 80)
print("PHASE 12 — CELL 1 COMPLETED")
print("=" * 80)

print("Total files:", len(DATASET_FILE_RECORDS))
print("Image files:", len(image_records))
print("Metadata files:", len(metadata_records))
print("Annotation candidates:", len(annotation_candidates))
print("Experimental masks:", len(experimental_mask_files))

print(
    "Reference images found:",
    int(reference_image_audit_df["image_file_found"].sum()),
    "/",
    len(REFERENCE_IDS)
)

print("\nEXPERT ANNOTATION STATUS: NOT YET CONFIRMED")
print("CLINICAL VALIDATION STATUS: NOT CLINICALLY VALIDATED")

print("\nSAVED REPORTS:")
print(inventory_path)
print(metadata_path)
print(annotation_path)
print(experimental_path)
print(reference_path)
print(summary_path)

print("\nNo source dataset files were modified.")
print("Share the complete cell output before proceeding to Cell 2.")


VELoura — PHASE 12: CLINICAL VALIDATION & DATASET READINESS
CELL 1: DATASET, IMAGE AND ANNOTATION INVENTORY
Mounted at /content/drive_phase12

Project directory: /content/drive_phase12/MyDrive/Veloura_Project
Dataset directory: /content/drive_phase12/MyDrive/Veloura_Project/datasets

DATASET INVENTORY
--------------------------------------------------------------------------------
Total dataset files: 9712
Image files: 9704
Metadata files: 4
Potential annotation-related files: 0

IMAGE FILE DISTRIBUTION


,extension,file_count
0,.jpg,9704



METADATA FILE SCHEMA


,file,status,columns,sample_rows
0,datasets/evaluation/extracted/manifest.json,PRESENT_NOT_TABULARLY_INSPECTED,None,NaN
1,datasets/evaluation/extracted/Health/eval/meta...,READ_SUCCESSFULLY,"[image_id, lesion_id, dx, age, sex, localization]",5.0
2,datasets/training/extracted/manifest.json,PRESENT_NOT_TABULARLY_INSPECTED,None,NaN
3,datasets/training/extracted/Health/training/me...,READ_SUCCESSFULLY,"[image_id, lesion_id, dx, age, sex, localization]",5.0



ANNOTATION DISCOVERY
No annotation-related filenames discovered.

IMPORTANT: A filename containing 'mask' or 'annotation' does not prove that the file is expert-annotated.

EXPERIMENTAL MASK INVENTORY
Experimental mask files: 24


,relative_path,filename,size_bytes,evidence_type,expert_reference
0,outputs/explainability_evidence/experimental_m...,ISIC_0025481_experimental_mask.png,2069,EXPERIMENTAL_VELOURA_GENERATED_MASK,False
1,outputs/explainability_evidence/experimental_m...,ISIC_0028145_experimental_mask.png,2160,EXPERIMENTAL_VELOURA_GENERATED_MASK,False
2,outputs/explainability_evidence/experimental_m...,ISIC_0031157_experimental_mask.png,2663,EXPERIMENTAL_VELOURA_GENERATED_MASK,False
3,outputs/explainability_evidence/experimental_m...,ISIC_0031437_experimental_mask.png,2282,EXPERIMENTAL_VELOURA_GENERATED_MASK,False
4,outputs/explainability_evidence/experimental_m...,ISIC_0033038_experimental_mask.png,2439,EXPERIMENTAL_VELOURA_GENERATED_MASK,False
5,outputs/explainability_evidence/experimental_m...,ISIC_0034119_experimental_mask.png,2188,EXPERIMENTAL_VELOURA_GENERATED_MASK,False
6,outputs/explainability_evidence/experimental_m...,ISIC_0025481_experimental_overlay.png,360302,EXPERIMENTAL_VELOURA_GENERATED_MASK,False
7,outputs/explainability_evidence/experimental_m...,ISIC_0028145_experimental_overlay.png,293947,EXPERIMENTAL_VELOURA_GENERATED_MASK,False
8,outputs/explainability_evidence/experimental_m...,ISIC_0031157_experimental_overlay.png,441090,EXPERIMENTAL_VELOURA_GENERATED_MASK,False
9,outputs/explainability_evidence/experimental_m...,ISIC_0031437_experimental_overlay.png,440531,EXPERIMENTAL_VELOURA_GENERATED_MASK,False



PHASE 5–11 REFERENCE IMAGE AUDIT


,image_id,image_file_found,matching_image_file_count,image_paths
0,ISIC_0025481,True,1,[datasets/training/extracted/Health/training/i...
1,ISIC_0028145,True,1,[datasets/training/extracted/Health/training/i...
2,ISIC_0031157,True,1,[datasets/training/extracted/Health/training/i...
3,ISIC_0031437,True,1,[datasets/training/extracted/Health/training/i...
4,ISIC_0033038,True,1,[datasets/training/extracted/Health/training/i...
5,ISIC_0034119,True,1,[datasets/training/extracted/Health/training/i...



PHASE 12 — CELL 1 COMPLETED
Total files: 9712
Image files: 9704
Metadata files: 4
Annotation candidates: 0
Experimental masks: 24
Reference images found: 6 / 6

EXPERT ANNOTATION STATUS: NOT YET CONFIRMED
CLINICAL VALIDATION STATUS: NOT CLINICALLY VALIDATED

SAVED REPORTS:
/content/drive_phase12/MyDrive/Veloura_Project/reports/clinical_validation/phase12_dataset_file_inventory.csv
/content/drive_phase12/MyDrive/Veloura_Project/reports/clinical_validation/phase12_metadata_schema_inventory.json
/content/drive_phase12/MyDrive/Veloura_Project/reports/clinical_validation/phase12_annotation_discovery.csv
/content/drive_phase12/MyDrive/Veloura_Project/reports/clinical_validation/phase12_experimental_mask_inventory.csv
/content/drive_phase12/MyDrive/Veloura_Project/reports/clinical_validation/phase12_reference_image_audit.csv
/content/drive_phase12/MyDrive/Veloura_Project/outputs/clinical_validation/phase12_dataset_inventory_summary.json

No source dataset files were modified.
Share the compl

In [ ]:

# ============================================================================
# VELOURA — PHASE 12, CELL 2
# FULL METADATA, REFERENCE-LABEL AND DATASET-SPLIT AUDIT
# ============================================================================

import pandas as pd
import json
from pathlib import Path
from datetime import datetime, timezone

print("=" * 80)
print("PHASE 12 — CELL 2: FULL METADATA & REFERENCE-LABEL AUDIT")
print("=" * 80)

# ---------------------------------------------------------------------------
# 1. Confirm Cell 1 prerequisites
# ---------------------------------------------------------------------------

if "PROJECT_DIR" not in globals():
    raise RuntimeError("Run Phase 12 Cell 1 first.")

PROJECT_DIR = Path(PROJECT_DIR)
DATASET_DIR = PROJECT_DIR / "datasets"

PHASE12_OUTPUT = PROJECT_DIR / "outputs" / "clinical_validation"
PHASE12_REPORTS = PROJECT_DIR / "reports" / "clinical_validation"

PHASE12_OUTPUT.mkdir(parents=True, exist_ok=True)
PHASE12_REPORTS.mkdir(parents=True, exist_ok=True)

REFERENCE_IDS = [
    "ISIC_0025481",
    "ISIC_0028145",
    "ISIC_0031157",
    "ISIC_0031437",
    "ISIC_0033038",
    "ISIC_0034119"
]

# ---------------------------------------------------------------------------
# 2. Discover metadata CSV files by their actual schema
# ---------------------------------------------------------------------------

metadata_candidates = []

for path in DATASET_DIR.rglob("*.csv"):

    try:
        header = pd.read_csv(path, nrows=0)

        if "image_id" in header.columns:
            metadata_candidates.append(path)

    except Exception as e:
        print("Skipped unreadable CSV:", path, "|", str(e)[:150])

if not metadata_candidates:
    raise RuntimeError(
        "No CSV metadata file containing an image_id column was found."
    )

print("\nMetadata CSV candidates:")
for path in metadata_candidates:
    print("-", path.relative_to(PROJECT_DIR))

# ---------------------------------------------------------------------------
# 3. Load complete metadata tables
# ---------------------------------------------------------------------------

metadata_tables = {}

for path in metadata_candidates:

    relative = str(path.relative_to(PROJECT_DIR)).lower()

    if "training" in relative:
        split = "training"
    elif "evaluation" in relative:
        split = "evaluation"
    else:
        split = "other"

    if split in metadata_tables:
        raise RuntimeError(
            f"Multiple metadata CSVs identified for split '{split}'. "
            "Review the candidate paths before continuing."
        )

    df = pd.read_csv(path, low_memory=False)

    # Preserve source values while normalizing whitespace in ID columns.
    for column in ["image_id", "lesion_id"]:
        if column in df.columns:
            df[column] = df[column].astype("string").str.strip()

    metadata_tables[split] = {
        "path": path,
        "data": df
    }

    print(
        f"\nLoaded {split}: {len(df):,} records | "
        f"{len(df.columns)} columns"
    )

    print("Columns:", list(df.columns))

if "training" not in metadata_tables:
    raise RuntimeError("Training metadata CSV was not identified.")

if "evaluation" not in metadata_tables:
    print(
        "\nWARNING: Evaluation metadata was not identified. "
        "Evaluation split comparisons will be limited."
    )

# ---------------------------------------------------------------------------
# 4. Generate per-dataset metadata summaries
# ---------------------------------------------------------------------------

summary_records = []
missing_records = []
label_records = []
duplicate_records = []

for split, item in metadata_tables.items():

    df = item["data"]

    summary_records.append({
        "dataset_split": split,
        "source_file": str(item["path"].relative_to(PROJECT_DIR)),
        "record_count": len(df),
        "column_count": len(df.columns),
        "unique_image_ids": (
            df["image_id"].nunique(dropna=True)
            if "image_id" in df.columns else None
        ),
        "unique_lesion_ids": (
            df["lesion_id"].nunique(dropna=True)
            if "lesion_id" in df.columns else None
        )
    })

    for column in df.columns:

        missing_records.append({
            "dataset_split": split,
            "column": column,
            "total_records": len(df),
            "missing_count": int(df[column].isna().sum()),
            "missing_percentage": (
                round(float(df[column].isna().mean() * 100), 2)
                if len(df) else 0
            )
        })

    if "dx" in df.columns:

        counts = (
            df["dx"]
            .fillna("<MISSING>")
            .astype(str)
            .value_counts(dropna=False)
        )

        for label, count in counts.items():

            label_records.append({
                "dataset_split": split,
                "source_label": label,
                "record_count": int(count)
            })

    for column in ["image_id", "lesion_id"]:

        if column not in df.columns:
            continue

        duplicated = df[column].duplicated(keep=False)
        nonmissing_duplicated = (
            duplicated & df[column].notna()
        )

        duplicate_records.append({
            "dataset_split": split,
            "identifier": column,
            "duplicate_row_count": int(nonmissing_duplicated.sum()),
            "duplicate_unique_values": int(
                df.loc[nonmissing_duplicated, column].nunique()
            )
        })

metadata_summary_df = pd.DataFrame(summary_records)
missingness_df = pd.DataFrame(missing_records)
label_distribution_df = pd.DataFrame(label_records)
duplicate_audit_df = pd.DataFrame(duplicate_records)

print("\nMETADATA DATASET SUMMARY")
display(metadata_summary_df)

print("\nMISSING VALUE AUDIT")
display(missingness_df)

print("\nSOURCE LABEL DISTRIBUTION")
display(label_distribution_df)

print("\nDUPLICATE IDENTIFIER AUDIT")
display(duplicate_audit_df)

# ---------------------------------------------------------------------------
# 5. Check training/evaluation image and lesion overlap
# ---------------------------------------------------------------------------

overlap_records = []
overlap_details = {}

if "training" in metadata_tables and "evaluation" in metadata_tables:

    train_df = metadata_tables["training"]["data"]
    eval_df = metadata_tables["evaluation"]["data"]

    for column in ["image_id", "lesion_id"]:

        if column not in train_df.columns or column not in eval_df.columns:
            overlap_records.append({
                "identifier": column,
                "overlap_count": None,
                "status": "COLUMN_UNAVAILABLE"
            })
            continue

        train_ids = set(
            train_df[column].dropna().astype(str)
        )

        eval_ids = set(
            eval_df[column].dropna().astype(str)
        )

        overlap = sorted(train_ids.intersection(eval_ids))

        overlap_details[column] = overlap

        overlap_records.append({
            "identifier": column,
            "training_unique_count": len(train_ids),
            "evaluation_unique_count": len(eval_ids),
            "overlap_count": len(overlap),
            "status": (
                "OVERLAP_DETECTED" if overlap
                else "NO_OVERLAP_DETECTED"
            )
        })

overlap_df = pd.DataFrame(overlap_records)

print("\nTRAINING / EVALUATION OVERLAP AUDIT")
display(overlap_df)

# ---------------------------------------------------------------------------
# 6. Extract metadata for the six Phase 5–11 reference images
# ---------------------------------------------------------------------------

reference_records = []

for split, item in metadata_tables.items():

    df = item["data"]

    if "image_id" not in df.columns:
        continue

    selected = df[df["image_id"].isin(REFERENCE_IDS)]

    for _, row in selected.iterrows():

        reference_records.append({
            "image_id": row.get("image_id"),
            "dataset_split": split,
            "lesion_id": row.get("lesion_id"),
            "dx_source_label": row.get("dx"),
            "age_source_value": row.get("age"),
            "sex_source_value": row.get("sex"),
            "localization_source_value": row.get("localization"),
            "metadata_source_file": str(
                item["path"].relative_to(PROJECT_DIR)
            )
        })

reference_metadata_df = pd.DataFrame(reference_records)

reference_coverage = []

for image_id in REFERENCE_IDS:

    matches = reference_metadata_df[
        reference_metadata_df["image_id"] == image_id
    ] if len(reference_metadata_df) else pd.DataFrame()

    reference_coverage.append({
        "image_id": image_id,
        "metadata_record_count": len(matches),
        "metadata_record_found": len(matches) > 0,
        "dataset_splits": (
            ";".join(sorted(matches["dataset_split"].unique()))
            if len(matches) else None
        ),
        "source_labels": (
            ";".join(
                sorted(
                    matches["dx_source_label"]
                    .dropna().astype(str).unique()
                )
            )
            if len(matches) else None
        )
    })

reference_coverage_df = pd.DataFrame(reference_coverage)

print("\nSIX REFERENCE CASES — METADATA RECORDS")
display(reference_metadata_df)

print("\nSIX REFERENCE CASES — COVERAGE")
display(reference_coverage_df)

# ---------------------------------------------------------------------------
# 7. Check age and metadata value consistency
# ---------------------------------------------------------------------------

value_quality_records = []

for split, item in metadata_tables.items():

    df = item["data"]

    if "age" in df.columns:

        age_numeric = pd.to_numeric(
            df["age"], errors="coerce"
        )

        value_quality_records.append({
            "dataset_split": split,
            "field": "age",
            "nonmissing_source_values": int(df["age"].notna().sum()),
            "numeric_values": int(age_numeric.notna().sum()),
            "non_numeric_nonmissing_values": int(
                (df["age"].notna() & age_numeric.isna()).sum()
            ),
            "minimum_numeric_value": (
                float(age_numeric.min())
                if age_numeric.notna().any() else None
            ),
            "maximum_numeric_value": (
                float(age_numeric.max())
                if age_numeric.notna().any() else None
            )
        })

    for column in ["sex", "localization"]:

        if column in df.columns:

            value_quality_records.append({
                "dataset_split": split,
                "field": column,
                "nonmissing_source_values": int(df[column].notna().sum()),
                "unique_nonmissing_values": int(
                    df[column].nunique(dropna=True)
                )
            })

value_quality_df = pd.DataFrame(value_quality_records)

print("\nMETADATA VALUE QUALITY")
display(value_quality_df)

# ---------------------------------------------------------------------------
# 8. Save Phase 12 audit outputs
# ---------------------------------------------------------------------------

reports = {
    "phase12_metadata_dataset_summary.csv": metadata_summary_df,
    "phase12_metadata_missingness.csv": missingness_df,
    "phase12_source_label_distribution.csv": label_distribution_df,
    "phase12_duplicate_identifier_audit.csv": duplicate_audit_df,
    "phase12_train_evaluation_overlap.csv": overlap_df,
    "phase12_reference_metadata.csv": reference_metadata_df,
    "phase12_reference_metadata_coverage.csv": reference_coverage_df,
    "phase12_metadata_value_quality.csv": value_quality_df
}

for filename, dataframe in reports.items():

    dataframe.to_csv(
        PHASE12_REPORTS / filename,
        index=False
    )

# Save overlap identifiers for traceability.
overlap_json_path = (
    PHASE12_OUTPUT / "phase12_split_overlap_details.json"
)

with open(overlap_json_path, "w", encoding="utf-8") as f:

    json.dump(
        overlap_details,
        f,
        indent=2,
        ensure_ascii=False
    )

# ---------------------------------------------------------------------------
# 9. Generate metadata audit summary
# ---------------------------------------------------------------------------

audit_summary = {
    "project": "Veloura Agentic AI 2026",
    "phase": 12,
    "cell": 2,
    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "metadata_files_loaded": len(metadata_tables),
    "dataset_summaries": summary_records,
    "reference_cases": len(REFERENCE_IDS),
    "reference_cases_with_metadata": int(
        reference_coverage_df["metadata_record_found"].sum()
    ),
    "train_evaluation_overlap": overlap_records,
    "expert_annotation_status": "NOT_CONFIRMED",
    "clinical_validation_status": "NOT_CLINICALLY_VALIDATED",
    "source_metadata_modified": False
}

summary_json_path = (
    PHASE12_OUTPUT / "phase12_metadata_audit_summary.json"
)

with open(summary_json_path, "w", encoding="utf-8") as f:
    json.dump(audit_summary, f, indent=2, ensure_ascii=False, default=str)

# ---------------------------------------------------------------------------
# 10. Final output
# ---------------------------------------------------------------------------

print("\n" + "=" * 80)
print("PHASE 12 — CELL 2 COMPLETED")
print("=" * 80)

print("Metadata datasets loaded:", len(metadata_tables))
print("Total metadata records:", sum(
    len(item["data"]) for item in metadata_tables.values()
))
print(
    "Reference cases with metadata:",
    int(reference_coverage_df["metadata_record_found"].sum()),
    "/ 6"
)

print("\nIMPORTANT:")
print("- Source labels are reported as provided, not clinically reinterpreted.")
print("- Metadata labels are not expert segmentation masks.")
print("- Dataset split overlap requires review before validation.")
print("- No clinical performance metric has been calculated.")

print("\nSaved reports:")
for filename in reports:
    print(PHASE12_REPORTS / filename)

print(summary_json_path)
print(overlap_json_path)

print("\nNo source metadata or image files were modified.")
print("Share the complete output before proceeding to Cell 3.")


PHASE 12 — CELL 2: FULL METADATA & REFERENCE-LABEL AUDIT

Metadata CSV candidates:
- datasets/evaluation/extracted/Health/eval/metadata.csv
- datasets/training/extracted/Health/training/metadata.csv

Loaded evaluation: 1,942 records | 6 columns
Columns: ['image_id', 'lesion_id', 'dx', 'age', 'sex', 'localization']

Loaded training: 7,762 records | 6 columns
Columns: ['image_id', 'lesion_id', 'dx', 'age', 'sex', 'localization']

METADATA DATASET SUMMARY


,dataset_split,source_file,record_count,column_count,unique_image_ids,unique_lesion_ids
0,evaluation,datasets/evaluation/extracted/Health/eval/meta...,1942,6,1942,1456
1,training,datasets/training/extracted/Health/training/me...,7762,6,7762,5815



MISSING VALUE AUDIT


,dataset_split,column,total_records,missing_count,missing_percentage
0,evaluation,image_id,1942,0,0.00
1,evaluation,lesion_id,1942,0,0.00
2,evaluation,dx,1942,0,0.00
3,evaluation,age,1942,12,0.62
4,evaluation,sex,1942,0,0.00
5,evaluation,localization,1942,0,0.00
6,training,image_id,7762,0,0.00
7,training,lesion_id,7762,0,0.00
8,training,dx,7762,0,0.00
9,training,age,7762,43,0.55



SOURCE LABEL DISTRIBUTION


,dataset_split,source_label,record_count
0,evaluation,nv,1302
1,evaluation,mel,214
2,evaluation,bkl,212
3,evaluation,bcc,101
4,evaluation,akiec,63
5,evaluation,vasc,27
6,evaluation,df,23
7,training,nv,5212
8,training,mel,856
9,training,bkl,848


... (7 outputs left out)

,image_id,metadata_record_count,metadata_record_found,dataset_splits,source_labels
0,ISIC_0025481,1,True,training,nv
1,ISIC_0028145,1,True,training,nv
2,ISIC_0031157,1,True,training,nv
3,ISIC_0031437,1,True,training,nv
4,ISIC_0033038,1,True,training,mel
5,ISIC_0034119,1,True,training,bcc



METADATA VALUE QUALITY


,dataset_split,field,nonmissing_source_values,numeric_values,non_numeric_nonmissing_values,minimum_numeric_value,maximum_numeric_value,unique_nonmissing_values
0,evaluation,age,1930,1930.0,0.0,0.0,85.0,NaN
1,evaluation,sex,1942,NaN,NaN,NaN,NaN,3.0
2,evaluation,localization,1942,NaN,NaN,NaN,NaN,15.0
3,training,age,7719,7719.0,0.0,0.0,85.0,NaN
4,training,sex,7762,NaN,NaN,NaN,NaN,3.0
5,training,localization,7762,NaN,NaN,NaN,NaN,15.0



PHASE 12 — CELL 2 COMPLETED
Metadata datasets loaded: 2
Total metadata records: 9704
Reference cases with metadata: 6 / 6

IMPORTANT:
- Source labels are reported as provided, not clinically reinterpreted.
- Metadata labels are not expert segmentation masks.
- Dataset split overlap requires review before validation.
- No clinical performance metric has been calculated.

Saved reports:
/content/drive_phase12/MyDrive/Veloura_Project/reports/clinical_validation/phase12_metadata_dataset_summary.csv
/content/drive_phase12/MyDrive/Veloura_Project/reports/clinical_validation/phase12_metadata_missingness.csv
/content/drive_phase12/MyDrive/Veloura_Project/reports/clinical_validation/phase12_source_label_distribution.csv
/content/drive_phase12/MyDrive/Veloura_Project/reports/clinical_validation/phase12_duplicate_identifier_audit.csv
/content/drive_phase12/MyDrive/Veloura_Project/reports/clinical_validation/phase12_train_evaluation_overlap.csv
/content/drive_phase12/MyDrive/Veloura_Project/repor

In [ ]:

# ============================================================================
# VELOURA — PHASE 12, CELL 3
# LESION-LEVEL CONSISTENCY, IMAGE MATCHING & DATASET PROVENANCE
# ============================================================================

import pandas as pd
import json
from pathlib import Path
from collections import defaultdict, Counter
from datetime import datetime, timezone

print("=" * 80)
print("PHASE 12 — CELL 3: LESION CONSISTENCY & DATASET PROVENANCE")
print("=" * 80)

# ---------------------------------------------------------------------------
# 1. Verify prerequisites and paths
# ---------------------------------------------------------------------------

if "PROJECT_DIR" not in globals():
    raise RuntimeError("Run Phase 12 Cell 1 first.")

PROJECT_DIR = Path(PROJECT_DIR)
DATASET_DIR = PROJECT_DIR / "datasets"

PHASE12_OUTPUT = PROJECT_DIR / "outputs" / "clinical_validation"
PHASE12_REPORTS = PROJECT_DIR / "reports" / "clinical_validation"

PHASE12_OUTPUT.mkdir(parents=True, exist_ok=True)
PHASE12_REPORTS.mkdir(parents=True, exist_ok=True)

TRAIN_METADATA_PATH = (
    DATASET_DIR
    / "training"
    / "extracted"
    / "Health"
    / "training"
    / "metadata.csv"
)

EVAL_METADATA_PATH = (
    DATASET_DIR
    / "evaluation"
    / "extracted"
    / "Health"
    / "eval"
    / "metadata.csv"
)

for path in [TRAIN_METADATA_PATH, EVAL_METADATA_PATH]:
    if not path.is_file():
        raise FileNotFoundError(path)

train_df = pd.read_csv(TRAIN_METADATA_PATH, low_memory=False)
eval_df = pd.read_csv(EVAL_METADATA_PATH, low_memory=False)

# ---------------------------------------------------------------------------
# 2. Lesion-level label consistency audit
# ---------------------------------------------------------------------------

def audit_lesion_consistency(df, split_name):

    required = {"image_id", "lesion_id", "dx"}

    if not required.issubset(df.columns):
        raise ValueError(
            f"Missing required columns in {split_name}: "
            f"{required - set(df.columns)}"
        )

    work = df.copy()

    work["lesion_id"] = (
        work["lesion_id"].astype("string").str.strip()
    )

    work["dx"] = (
        work["dx"].astype("string").str.strip()
    )

    records = []

    for lesion_id, group in work.groupby(
        "lesion_id", dropna=False
    ):

        labels = sorted(
            group["dx"].dropna().astype(str).unique().tolist()
        )

        records.append({
            "dataset_split": split_name,
            "lesion_id": str(lesion_id),
            "image_count": len(group),
            "unique_source_label_count": len(labels),
            "source_labels": ";".join(labels),
            "label_consistent": len(labels) <= 1,
            "image_ids": ";".join(
                group["image_id"].dropna().astype(str).tolist()
            )
        })

    return pd.DataFrame(records)

train_lesion_audit = audit_lesion_consistency(
    train_df, "training"
)

eval_lesion_audit = audit_lesion_consistency(
    eval_df, "evaluation"
)

lesion_audit_df = pd.concat(
    [train_lesion_audit, eval_lesion_audit],
    ignore_index=True
)

# ---------------------------------------------------------------------------
# 3. Summarize lesion grouping and label consistency
# ---------------------------------------------------------------------------

lesion_summary_records = []

for split_name, audit_df in [
    ("training", train_lesion_audit),
    ("evaluation", eval_lesion_audit)
]:

    lesion_summary_records.append({
        "dataset_split": split_name,
        "unique_lesions": len(audit_df),
        "single_image_lesions": int(
            (audit_df["image_count"] == 1).sum()
        ),
        "multiple_image_lesions": int(
            (audit_df["image_count"] > 1).sum()
        ),
        "maximum_images_per_lesion": int(
            audit_df["image_count"].max()
        ),
        "label_consistent_lesions": int(
            audit_df["label_consistent"].sum()
        ),
        "label_inconsistent_lesions": int(
            (~audit_df["label_consistent"]).sum()
        )
    })

lesion_summary_df = pd.DataFrame(lesion_summary_records)

inconsistent_lesions_df = lesion_audit_df[
    ~lesion_audit_df["label_consistent"]
].copy()

print("\nLESION GROUPING SUMMARY")
display(lesion_summary_df)

print("\nLESIONS WITH INCONSISTENT SOURCE LABELS")

if len(inconsistent_lesions_df):
    display(inconsistent_lesions_df)
else:
    print("No conflicting source labels detected within lesion groups.")

# ---------------------------------------------------------------------------
# 4. Build image-ID to physical-file index
# ---------------------------------------------------------------------------

image_extensions = {
    ".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"
}

image_file_index = defaultdict(list)

for path in DATASET_DIR.rglob("*"):

    if not path.is_file():
        continue

    if path.suffix.lower() not in image_extensions:
        continue

    image_id = path.stem.split(".")[0].strip()

    if image_id:
        image_file_index[image_id].append(path)

print("\nPhysical image files indexed:", sum(
    len(paths) for paths in image_file_index.values()
))

# ---------------------------------------------------------------------------
# 5. Match metadata image IDs against physical image files
# ---------------------------------------------------------------------------

def match_images_to_metadata(df, split_name):

    records = []

    for _, row in df.iterrows():

        image_id = str(row["image_id"]).strip()

        matches = image_file_index.get(image_id, [])

        expected_split_keyword = (
            "training" if split_name == "training" else "eval"
        )

        split_matches = [
            p for p in matches
            if expected_split_keyword in [
                part.lower() for part in p.parts
            ]
        ]

        records.append({
            "dataset_split": split_name,
            "image_id": image_id,
            "physical_file_found": len(matches) > 0,
            "matching_file_count": len(matches),
            "matching_split_file_count": len(split_matches),
            "correct_split_file_found": len(split_matches) > 0,
            "matched_paths": ";".join(
                str(p.relative_to(PROJECT_DIR))
                for p in matches
            )
        })

    return pd.DataFrame(records)

train_image_match_df = match_images_to_metadata(
    train_df, "training"
)

eval_image_match_df = match_images_to_metadata(
    eval_df, "evaluation"
)

image_match_df = pd.concat(
    [train_image_match_df, eval_image_match_df],
    ignore_index=True
)

image_match_summary_df = (
    image_match_df
    .groupby("dataset_split")
    .agg(
        metadata_records=("image_id", "count"),
        images_found=("physical_file_found", "sum"),
        correct_split_images=("correct_split_file_found", "sum"),
        duplicate_physical_matches=(
            "matching_file_count",
            lambda x: int((x > 1).sum())
        )
    )
    .reset_index()
)

print("\nIMAGE-TO-METADATA MATCH SUMMARY")
display(image_match_summary_df)

unmatched_images_df = image_match_df[
    ~image_match_df["correct_split_file_found"]
].copy()

duplicate_image_matches_df = image_match_df[
    image_match_df["matching_file_count"] > 1
].copy()

print("\nUNMATCHED OR WRONG-SPLIT IMAGES:", len(unmatched_images_df))

if len(unmatched_images_df):
    display(unmatched_images_df.head(30))

print("\nIMAGE IDs WITH MULTIPLE PHYSICAL FILE MATCHES:",
      len(duplicate_image_matches_df))

if len(duplicate_image_matches_df):
    display(duplicate_image_matches_df.head(30))

# ---------------------------------------------------------------------------
# 6. Inspect dataset manifests
# ---------------------------------------------------------------------------

manifest_paths = [
    DATASET_DIR / "training" / "extracted" / "manifest.json",
    DATASET_DIR / "evaluation" / "extracted" / "manifest.json"
]

manifest_records = []

for manifest_path in manifest_paths:

    record = {
        "manifest_path": str(
            manifest_path.relative_to(PROJECT_DIR)
        ),
        "exists": manifest_path.is_file(),
        "read_status": None,
        "top_level_type": None,
        "top_level_keys": None,
        "content_preview": None
    }

    if manifest_path.is_file():

        try:
            with open(
                manifest_path, "r", encoding="utf-8"
            ) as f:
                manifest_data = json.load(f)

            record["read_status"] = "READ_SUCCESSFULLY"
            record["top_level_type"] = type(
                manifest_data
            ).__name__

            if isinstance(manifest_data, dict):
                record["top_level_keys"] = list(
                    manifest_data.keys()
                )

            record["content_preview"] = str(
                manifest_data
            )[:1500]

        except Exception as e:
            record["read_status"] = "READ_ERROR"
            record["content_preview"] = str(e)[:500]

    else:
        record["read_status"] = "NOT_FOUND"

    manifest_records.append(record)

manifest_audit_df = pd.DataFrame(manifest_records)

print("\nDATASET MANIFEST AUDIT")
display(manifest_audit_df)

# ---------------------------------------------------------------------------
# 7. Search for archive files and dataset documentation
# ---------------------------------------------------------------------------

documentation_extensions = {
    ".txt", ".md", ".pdf", ".yaml", ".yml", ".json"
}

documentation_keywords = [
    "readme", "license", "citation", "dataset",
    "provenance", "source", "description", "metadata"
]

documentation_records = []
archive_records = []

for path in PROJECT_DIR.rglob("*"):

    if not path.is_file():
        continue

    name_lower = path.name.lower()

    if path.suffix.lower() in documentation_extensions:
        if any(
            keyword in name_lower
            for keyword in documentation_keywords
        ):
            documentation_records.append({
                "relative_path": str(
                    path.relative_to(PROJECT_DIR)
                ),
                "filename": path.name,
                "size_bytes": path.stat().st_size
            })

    if path.suffix.lower() in {
        ".zip", ".tar", ".gz", ".7z"
    }:
        archive_records.append({
            "relative_path": str(
                path.relative_to(PROJECT_DIR)
            ),
            "filename": path.name,
            "size_bytes": path.stat().st_size
        })

documentation_df = pd.DataFrame(documentation_records)
archive_df = pd.DataFrame(archive_records)

print("\nPOTENTIAL DATASET DOCUMENTATION")
if len(documentation_df):
    display(documentation_df)
else:
    print("No matching documentation files discovered.")

print("\nARCHIVE INVENTORY")
if len(archive_df):
    display(archive_df)
else:
    print("No archive files discovered in the project directory.")

# ---------------------------------------------------------------------------
# 8. Create data-split suitability flags
# ---------------------------------------------------------------------------

split_suitability_records = []

for split_name, audit_df, match_df in [
    ("training", train_lesion_audit, train_image_match_df),
    ("evaluation", eval_lesion_audit, eval_image_match_df)
]:

    label_consistency = bool(
        audit_df["label_consistent"].all()
    )

    image_mapping_complete = bool(
        match_df["correct_split_file_found"].all()
    )

    split_suitability_records.append({
        "dataset_split": split_name,
        "lesion_label_consistency": label_consistency,
        "image_mapping_complete": image_mapping_complete,
        "suitable_for_final_clinical_validation": False,
        "reason": (
            "Expert annotation, label provenance and validation "
            "protocol still require confirmation."
        )
    })

split_suitability_df = pd.DataFrame(
    split_suitability_records
)

print("\nDATASET SPLIT READINESS")
display(split_suitability_df)

# ---------------------------------------------------------------------------
# 9. Save all Phase 12 Cell 3 outputs
# ---------------------------------------------------------------------------

reports = {
    "phase12_lesion_level_audit.csv": lesion_audit_df,
    "phase12_lesion_consistency_summary.csv": lesion_summary_df,
    "phase12_inconsistent_lesions.csv": inconsistent_lesions_df,
    "phase12_image_metadata_matching.csv": image_match_df,
    "phase12_image_matching_summary.csv": image_match_summary_df,
    "phase12_unmatched_images.csv": unmatched_images_df,
    "phase12_duplicate_image_matches.csv": duplicate_image_matches_df,
    "phase12_manifest_audit.csv": manifest_audit_df,
    "phase12_dataset_documentation_inventory.csv": documentation_df,
    "phase12_archive_inventory.csv": archive_df,
    "phase12_split_suitability.csv": split_suitability_df
}

for filename, df in reports.items():
    df.to_csv(
        PHASE12_REPORTS / filename,
        index=False
    )

# ---------------------------------------------------------------------------
# 10. Save machine-readable summary
# ---------------------------------------------------------------------------

cell3_summary = {
    "project": "Veloura Agentic AI 2026",
    "phase": 12,
    "cell": 3,
    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "training_lesions": len(train_lesion_audit),
    "evaluation_lesions": len(eval_lesion_audit),
    "training_inconsistent_lesions": len(
        train_lesion_audit[
            ~train_lesion_audit["label_consistent"]
        ]
    ),
    "evaluation_inconsistent_lesions": len(
        eval_lesion_audit[
            ~eval_lesion_audit["label_consistent"]
        ]
    ),
    "training_images_correctly_mapped": int(
        train_image_match_df["correct_split_file_found"].sum()
    ),
    "training_metadata_records": len(train_image_match_df),
    "evaluation_images_correctly_mapped": int(
        eval_image_match_df["correct_split_file_found"].sum()
    ),
    "evaluation_metadata_records": len(eval_image_match_df),
    "documentation_candidates": len(documentation_df),
    "archive_files": len(archive_df),
    "expert_annotation_status": "NOT_CONFIRMED",
    "clinical_validation_status": "NOT_CLINICALLY_VALIDATED",
    "source_dataset_modified": False
}

summary_path = (
    PHASE12_OUTPUT / "phase12_lesion_and_provenance_summary.json"
)

with open(summary_path, "w", encoding="utf-8") as f:
    json.dump(
        cell3_summary,
        f,
        indent=2,
        ensure_ascii=False
    )

# ---------------------------------------------------------------------------
# 11. Final output
# ---------------------------------------------------------------------------

print("\n" + "=" * 80)
print("PHASE 12 — CELL 3 COMPLETED")
print("=" * 80)

print("Training lesion groups:", len(train_lesion_audit))
print("Evaluation lesion groups:", len(eval_lesion_audit))

print(
    "Inconsistent training lesion groups:",
    len(inconsistent_lesions_df[
        inconsistent_lesions_df["dataset_split"] == "training"
    ])
)

print(
    "Inconsistent evaluation lesion groups:",
    len(inconsistent_lesions_df[
        inconsistent_lesions_df["dataset_split"] == "evaluation"
    ])
)

print(
    "Training images mapped:",
    int(train_image_match_df["correct_split_file_found"].sum()),
    "/",
    len(train_image_match_df)
)

print(
    "Evaluation images mapped:",
    int(eval_image_match_df["correct_split_file_found"].sum()),
    "/",
    len(eval_image_match_df)
)

print("\nExpert annotations remain unconfirmed.")
print("No clinical performance metrics calculated.")
print("No source dataset files modified.")

print("\nSaved", len(reports), "CSV reports.")
print(summary_path)

print("\nShare the complete output before proceeding to Cell 4.")


PHASE 12 — CELL 3: LESION CONSISTENCY & DATASET PROVENANCE

LESION GROUPING SUMMARY


,dataset_split,unique_lesions,single_image_lesions,multiple_image_lesions,maximum_images_per_lesion,label_consistent_lesions,label_inconsistent_lesions
0,training,5815,4317,1498,6,5815,0
1,evaluation,1456,1082,374,6,1456,0



LESIONS WITH INCONSISTENT SOURCE LABELS
No conflicting source labels detected within lesion groups.

Physical image files indexed: 9704

IMAGE-TO-METADATA MATCH SUMMARY


,dataset_split,metadata_records,images_found,correct_split_images,duplicate_physical_matches
0,evaluation,1942,1942,1942,0
1,training,7762,7762,7762,0



UNMATCHED OR WRONG-SPLIT IMAGES: 0

IMAGE IDs WITH MULTIPLE PHYSICAL FILE MATCHES: 0

DATASET MANIFEST AUDIT


,manifest_path,exists,read_status,top_level_type,top_level_keys,content_preview
0,datasets/training/extracted/manifest.json,True,READ_SUCCESSFULLY,dict,"[pack, source, included_partitions, excluded, ...","{'pack': 'participant', 'source': 'DataSet/{Cy..."
1,datasets/evaluation/extracted/manifest.json,True,READ_SUCCESSFULLY,dict,"[pack, source, included_partitions, excluded, ...","{'pack': 'participant', 'source': 'DataSet/{Cy..."


... (3 outputs left out)

,relative_path,filename,size_bytes
0,datasets/training/health-training.zip,health-training.zip,2153898562
1,datasets/evaluation/health-eval.zip,health-eval.zip,533958185



DATASET SPLIT READINESS


,dataset_split,lesion_label_consistency,image_mapping_complete,suitable_for_final_clinical_validation,reason
0,training,True,True,False,"Expert annotation, label provenance and valida..."
1,evaluation,True,True,False,"Expert annotation, label provenance and valida..."



PHASE 12 — CELL 3 COMPLETED
Training lesion groups: 5815
Evaluation lesion groups: 1456
Inconsistent training lesion groups: 0
Inconsistent evaluation lesion groups: 0
Training images mapped: 7762 / 7762
Evaluation images mapped: 1942 / 1942

Expert annotations remain unconfirmed.
No clinical performance metrics calculated.
No source dataset files modified.

Saved 11 CSV reports.
/content/drive_phase12/MyDrive/Veloura_Project/outputs/clinical_validation/phase12_lesion_and_provenance_summary.json

Share the complete output before proceeding to Cell 4.


In [ ]:
# =============================================================================
# PHASE 12 — CELL 4: DOCUMENTATION, LABEL & ANNOTATION PROVENANCE REVIEW
# =============================================================================

import os
import json
import pandas as pd
from pathlib import Path
from datetime import datetime, timezone

PROJECT_DIR = Path(
    "/content/drive_phase12/MyDrive/Veloura_Project"
)

OUTPUT_DIR = PROJECT_DIR / "outputs" / "clinical_validation"
REPORT_DIR = PROJECT_DIR / "reports" / "clinical_validation"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)

# -------------------------------------------------------------------------
# 1. Read dataset documentation
# -------------------------------------------------------------------------

documentation_paths = [
    PROJECT_DIR / "README.md",
    PROJECT_DIR / "datasets/training/extracted/README.md",
    PROJECT_DIR / "datasets/evaluation/extracted/README.md",
]

documentation_records = []

for path in documentation_paths:
    record = {
        "relative_path": str(path.relative_to(PROJECT_DIR)),
        "exists": path.exists(),
        "read_status": "NOT_READ",
        "content": ""
    }

    if path.exists():
        try:
            content = path.read_text(
                encoding="utf-8",
                errors="replace"
            )

            record["content"] = content
            record["read_status"] = "READ_SUCCESSFULLY"

            print("\n" + "=" * 80)
            print(record["relative_path"])
            print("=" * 80)
            print(content[:12000])

        except Exception as e:
            record["read_status"] = f"READ_ERROR: {str(e)}"

    documentation_records.append(record)

# -------------------------------------------------------------------------
# 2. Read full dataset manifests
# -------------------------------------------------------------------------

manifest_paths = [
    PROJECT_DIR / "datasets/training/extracted/manifest.json",
    PROJECT_DIR / "datasets/evaluation/extracted/manifest.json",
]

manifest_records = []

for path in manifest_paths:
    record = {
        "relative_path": str(path.relative_to(PROJECT_DIR)),
        "exists": path.exists(),
        "read_status": "NOT_READ",
        "content": {}
    }

    if path.exists():
        try:
            with open(path, "r", encoding="utf-8") as f:
                content = json.load(f)

            record["content"] = content
            record["read_status"] = "READ_SUCCESSFULLY"

            print("\n" + "=" * 80)
            print(record["relative_path"])
            print("=" * 80)
            print(json.dumps(content, indent=2, default=str)[:16000])

        except Exception as e:
            record["read_status"] = f"READ_ERROR: {str(e)}"

    manifest_records.append(record)

# -------------------------------------------------------------------------
# 3. Search documentation for provenance and annotation statements
# -------------------------------------------------------------------------

keywords = [
    "source", "dataset", "license", "citation", "isic",
    "ground truth", "ground-truth", "annotation",
    "annotated", "segmentation", "mask", "expert",
    "diagnosis", "diagnostic", "label", "metadata",
    "training", "evaluation", "patient", "lesion"
]

keyword_records = []

for doc in documentation_records:
    content = doc["content"]

    for line_number, line in enumerate(content.splitlines(), start=1):
        matched = [
            keyword for keyword in keywords
            if keyword.lower() in line.lower()
        ]

        if matched:
            keyword_records.append({
                "document": doc["relative_path"],
                "line_number": line_number,
                "matched_keywords": ";".join(matched),
                "text": line.strip()[:1000]
            })

keyword_df = pd.DataFrame(keyword_records)

# -------------------------------------------------------------------------
# 4. Build a conservative provenance assessment
# -------------------------------------------------------------------------

documentation_readable = all(
    doc["read_status"] == "READ_SUCCESSFULLY"
    for doc in documentation_records
    if doc["exists"]
)

manifests_readable = all(
    item["read_status"] == "READ_SUCCESSFULLY"
    for item in manifest_records
    if item["exists"]
)

provenance_summary = {
    "phase": 12,
    "cell": 4,
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "documentation_files": [
        {
            "path": d["relative_path"],
            "exists": d["exists"],
            "read_status": d["read_status"],
            "character_count": len(d["content"])
        }
        for d in documentation_records
    ],
    "manifest_files": [
        {
            "path": m["relative_path"],
            "exists": m["exists"],
            "read_status": m["read_status"],
            "top_level_keys": list(m["content"].keys())
            if isinstance(m["content"], dict) else []
        }
        for m in manifest_records
    ],
    "documentation_readable": documentation_readable,
    "manifests_readable": manifests_readable,
    "source_label_provenance": "REQUIRES_REVIEW",
    "expert_segmentation_annotation_provenance": "UNCONFIRMED",
    "clinical_ground_truth_status": "NOT_ESTABLISHED",
    "clinical_validation_status": "NOT_CLINICALLY_VALIDATED",
    "automatic_acceptance": False,
    "independent_diagnosis": False,
    "mandatory_human_review": True
}

# -------------------------------------------------------------------------
# 5. Save outputs
# -------------------------------------------------------------------------

documentation_df = pd.DataFrame([
    {
        "relative_path": d["relative_path"],
        "exists": d["exists"],
        "read_status": d["read_status"],
        "character_count": len(d["content"])
    }
    for d in documentation_records
])

manifest_df = pd.DataFrame([
    {
        "relative_path": m["relative_path"],
        "exists": m["exists"],
        "read_status": m["read_status"],
        "top_level_keys": ";".join(m["content"].keys())
        if isinstance(m["content"], dict) else ""
    }
    for m in manifest_records
])

documentation_df.to_csv(
    REPORT_DIR / "phase12_documentation_read_audit.csv",
    index=False
)

manifest_df.to_csv(
    REPORT_DIR / "phase12_full_manifest_read_audit.csv",
    index=False
)

keyword_df.to_csv(
    REPORT_DIR / "phase12_provenance_keyword_evidence.csv",
    index=False
)

with open(
    OUTPUT_DIR / "phase12_documentation_provenance_summary.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(provenance_summary, f, indent=2)

print("\n" + "=" * 80)
print("PHASE 12 — CELL 4 COMPLETED")
print("=" * 80)

print("Documentation files read:",
      sum(d["read_status"] == "READ_SUCCESSFULLY"
          for d in documentation_records))

print("Manifest files read:",
      sum(m["read_status"] == "READ_SUCCESSFULLY"
          for m in manifest_records))

print("Provenance evidence lines:", len(keyword_df))
print("Source label provenance: REQUIRES_REVIEW")
print("Expert annotations: UNCONFIRMED")
print("Clinical validation: NOT_CLINICALLY_VALIDATED")
print("No source dataset files modified.")

print("\nSaved reports:")
print(REPORT_DIR / "phase12_documentation_read_audit.csv")
print(REPORT_DIR / "phase12_full_manifest_read_audit.csv")
print(REPORT_DIR / "phase12_provenance_keyword_evidence.csv")
print(OUTPUT_DIR / "phase12_documentation_provenance_summary.json")


README.md

# VELOURA
## Agentic DermaGuard
### Multi-Agent Skin Lesion Screening & Triage

Veloura is an explainable multi-agent research prototype
for dermoscopic skin-lesion classification and screening triage.

## Six Agents

1. Image Quality Agent
2. Diagnosis Agent
3. Lesion Analysis Agent
4. Reasoning & Context Agent
5. Safety Agent
6. Coordinator Agent

## Triage Decisions

- AUTO_TRIAGE
- REFER
- RE_IMAGE

## Technology Stack

- Python
- PyTorch
- OpenCV
- Scikit-learn
- Streamlit
- Google Colab

## Development Status

Phase 1: Environment Setup - In Progress

## Important Disclaimer

Veloura is a research and demonstration prototype.
It is not a clinically validated diagnostic system
and must not be used to make medical decisions.


datasets/training/extracted/README.md
# Participant Data Pack

This directory contains the data released to participants. Every track is split into
`training/` and `eval/`. The hidden evaluation inputs are **not** included here; the
organizers rel

In [ ]:
# =============================================================================
# PHASE 12 — CELL 5: MANIFEST INTEGRITY & DATASET HASH VERIFICATION
# =============================================================================

import hashlib
import json
import pandas as pd
from pathlib import Path
from datetime import datetime, timezone

PROJECT_DIR = Path(
    "/content/drive_phase12/MyDrive/Veloura_Project"
)

OUTPUT_DIR = PROJECT_DIR / "outputs" / "clinical_validation"
REPORT_DIR = PROJECT_DIR / "reports" / "clinical_validation"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)

def sha256_file(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()

    with open(path, "rb") as f:
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)

    return digest.hexdigest()

manifest_path = (
    PROJECT_DIR / "datasets/training/extracted/manifest.json"
)

with open(manifest_path, "r", encoding="utf-8") as f:
    manifest = json.load(f)

health = manifest["tracks"]["health"]

integrity_records = []

for split in ["training", "eval"]:
    split_info = health["partitions"][split]

    metadata_path = (
        PROJECT_DIR / "datasets" /
        ("training" if split == "training" else "evaluation") /
        "extracted" / "Health" /
        ("training" if split == "training" else "eval") /
        "metadata.csv"
    )

    actual_hash = (
        sha256_file(metadata_path)
        if metadata_path.exists() else None
    )

    expected_hash = split_info.get("metadata_sha256")

    integrity_records.append({
        "dataset_split": split,
        "metadata_path": str(metadata_path.relative_to(PROJECT_DIR)),
        "file_exists": metadata_path.exists(),
        "expected_sha256": expected_hash,
        "actual_sha256": actual_hash,
        "hash_match": (
            actual_hash == expected_hash
            if actual_hash and expected_hash else False
        ),
        "manifest_image_count": split_info.get("images"),
        "integrity_status": (
            "HASH_MATCH"
            if actual_hash and expected_hash == actual_hash
            else "HASH_MISMATCH_OR_UNAVAILABLE"
        )
    })

integrity_df = pd.DataFrame(integrity_records)

# -------------------------------------------------------------------------
# Verify metadata row counts and image file counts
# -------------------------------------------------------------------------

count_records = []

for split in ["training", "eval"]:
    base = (
        PROJECT_DIR / "datasets" /
        ("training" if split == "training" else "evaluation") /
        "extracted" / "Health" /
        ("training" if split == "training" else "eval")
    )

    metadata_path = base / "metadata.csv"
    image_dir = base / "images"

    metadata = pd.read_csv(metadata_path)

    image_count = sum(
        1 for p in image_dir.iterdir()
        if p.is_file() and p.suffix.lower() in
        [".jpg", ".jpeg", ".png"]
    )

    manifest_count = health["partitions"][split]["images"]

    count_records.append({
        "dataset_split": split,
        "metadata_rows": len(metadata),
        "unique_image_ids": metadata["image_id"].nunique(),
        "physical_image_files": image_count,
        "manifest_image_count": manifest_count,
        "metadata_count_matches_manifest": (
            len(metadata) == manifest_count
        ),
        "physical_count_matches_manifest": (
            image_count == manifest_count
        ),
        "unique_ids_match_rows": (
            metadata["image_id"].nunique() == len(metadata)
        )
    })

count_df = pd.DataFrame(count_records)

# -------------------------------------------------------------------------
# Verify the two metadata files are distinct
# -------------------------------------------------------------------------

train_hash = integrity_df.loc[
    integrity_df["dataset_split"] == "training",
    "actual_sha256"
].iloc[0]

eval_hash = integrity_df.loc[
    integrity_df["dataset_split"] == "eval",
    "actual_sha256"
].iloc[0]

metadata_files_distinct = (
    train_hash is not None and
    eval_hash is not None and
    train_hash != eval_hash
)

# -------------------------------------------------------------------------
# Final integrity status
# -------------------------------------------------------------------------

hashes_pass = bool(integrity_df["hash_match"].all())

counts_pass = bool(
    count_df[
        [
            "metadata_count_matches_manifest",
            "physical_count_matches_manifest",
            "unique_ids_match_rows"
        ]
    ].all().all()
)

integrity_status = (
    "DATASET_INTEGRITY_CHECKS_PASSED"
    if hashes_pass and counts_pass and metadata_files_distinct
    else "DATASET_INTEGRITY_REVIEW_REQUIRED"
)

summary = {
    "phase": 12,
    "cell": 5,
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "metadata_hash_verification": integrity_records,
    "split_count_verification": count_records,
    "metadata_files_distinct": metadata_files_distinct,
    "all_metadata_hashes_match": hashes_pass,
    "all_count_checks_pass": counts_pass,
    "integrity_status": integrity_status,
    "expert_annotations": "UNCONFIRMED",
    "clinical_validation": "NOT_CLINICALLY_VALIDATED",
    "automatic_acceptance": False,
    "independent_diagnosis": False,
    "mandatory_human_review": True
}

# -------------------------------------------------------------------------
# Save reports
# -------------------------------------------------------------------------

integrity_df.to_csv(
    REPORT_DIR / "phase12_metadata_hash_verification.csv",
    index=False
)

count_df.to_csv(
    REPORT_DIR / "phase12_manifest_count_verification.csv",
    index=False
)

with open(
    OUTPUT_DIR / "phase12_dataset_integrity_summary.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(summary, f, indent=2)

print("\n" + "=" * 80)
print("PHASE 12 — CELL 5 RESULTS")
print("=" * 80)

print("\nMETADATA HASH VERIFICATION")
display(integrity_df)

print("\nMANIFEST COUNT VERIFICATION")
display(count_df)

print("\nMetadata files distinct:", metadata_files_distinct)
print("All metadata hashes match:", hashes_pass)
print("All count checks pass:", counts_pass)
print("Integrity status:", integrity_status)

print("\nExpert annotations: UNCONFIRMED")
print("Clinical validation: NOT_CLINICALLY_VALIDATED")
print("No source dataset files modified.")

print("\nSaved:")
print(REPORT_DIR / "phase12_metadata_hash_verification.csv")
print(REPORT_DIR / "phase12_manifest_count_verification.csv")
print(OUTPUT_DIR / "phase12_dataset_integrity_summary.json")


PHASE 12 — CELL 5 RESULTS

METADATA HASH VERIFICATION


,dataset_split,metadata_path,file_exists,expected_sha256,actual_sha256,hash_match,manifest_image_count,integrity_status
0,training,datasets/training/extracted/Health/training/me...,True,5d5bdff86628882203b8e250d9c5e4aa13839456a9da73...,5d5bdff86628882203b8e250d9c5e4aa13839456a9da73...,True,7762,HASH_MATCH
1,eval,datasets/evaluation/extracted/Health/eval/meta...,True,1a7ab2121e3a36d6fa1ee1298e9a313607dde9509f6809...,1a7ab2121e3a36d6fa1ee1298e9a313607dde9509f6809...,True,1942,HASH_MATCH



MANIFEST COUNT VERIFICATION


,dataset_split,metadata_rows,unique_image_ids,physical_image_files,manifest_image_count,metadata_count_matches_manifest,physical_count_matches_manifest,unique_ids_match_rows
0,training,7762,7762,7762,7762,True,True,True
1,eval,1942,1942,1942,1942,True,True,True



Metadata files distinct: True
All metadata hashes match: True
All count checks pass: True
Integrity status: DATASET_INTEGRITY_CHECKS_PASSED

Expert annotations: UNCONFIRMED
Clinical validation: NOT_CLINICALLY_VALIDATED
No source dataset files modified.

Saved:
/content/drive_phase12/MyDrive/Veloura_Project/reports/clinical_validation/phase12_metadata_hash_verification.csv
/content/drive_phase12/MyDrive/Veloura_Project/reports/clinical_validation/phase12_manifest_count_verification.csv
/content/drive_phase12/MyDrive/Veloura_Project/outputs/clinical_validation/phase12_dataset_integrity_summary.json


In [ ]:
# =============================================================================
# PHASE 12 — CELL 6: CLINICAL VALIDATION PROTOCOL & METRIC READINESS
# =============================================================================

import json
import pandas as pd
from pathlib import Path
from datetime import datetime, timezone

PROJECT_DIR = Path(
    "/content/drive_phase12/MyDrive/Veloura_Project"
)

OUTPUT_DIR = PROJECT_DIR / "outputs" / "clinical_validation"
REPORT_DIR = PROJECT_DIR / "reports" / "clinical_validation"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)

# -------------------------------------------------------------------------
# 1. Define required clinical evidence and metric readiness
# -------------------------------------------------------------------------

metric_records = [
    {
        "metric": "Dice coefficient",
        "task": "Lesion segmentation",
        "required_reference": "Verified expert pixel-level segmentation masks",
        "evidence_available": False,
        "readiness": "NOT_READY",
        "reason": "Expert reference masks have not been confirmed."
    },
    {
        "metric": "Intersection over Union (IoU)",
        "task": "Lesion segmentation",
        "required_reference": "Verified expert pixel-level segmentation masks",
        "evidence_available": False,
        "readiness": "NOT_READY",
        "reason": "Expert reference masks have not been confirmed."
    },
    {
        "metric": "Sensitivity",
        "task": "Classification",
        "required_reference": "Verified reference labels and held-out model predictions",
        "evidence_available": False,
        "readiness": "NOT_READY",
        "reason": "No independently validated prediction set is available."
    },
    {
        "metric": "Specificity",
        "task": "Classification",
        "required_reference": "Verified reference labels and held-out model predictions",
        "evidence_available": False,
        "readiness": "NOT_READY",
        "reason": "No independently validated prediction set is available."
    },
    {
        "metric": "ROC-AUC",
        "task": "Classification",
        "required_reference": "Verified reference labels and model probability outputs",
        "evidence_available": False,
        "readiness": "NOT_READY",
        "reason": "Validated reference labels and probability outputs are not available."
    },
    {
        "metric": "Precision",
        "task": "Classification",
        "required_reference": "Verified reference labels and held-out model predictions",
        "evidence_available": False,
        "readiness": "NOT_READY",
        "reason": "No independently validated prediction set is available."
    },
    {
        "metric": "F1-score",
        "task": "Classification",
        "required_reference": "Verified reference labels and held-out model predictions",
        "evidence_available": False,
        "readiness": "NOT_READY",
        "reason": "No independently validated prediction set is available."
    },
    {
        "metric": "Calibration",
        "task": "Classification",
        "required_reference": "Verified labels and representative predicted probabilities",
        "evidence_available": False,
        "readiness": "NOT_READY",
        "reason": "Clinical reference labels and validated probability outputs are not available."
    }
]

metric_df = pd.DataFrame(metric_records)

# -------------------------------------------------------------------------
# 2. Define clinical validation requirements
# -------------------------------------------------------------------------

protocol_records = [
    {
        "requirement_id": "CV01",
        "requirement": "Verified reference labels",
        "description": "Document label definitions, provenance and qualified reference review.",
        "current_status": "REQUIRES_REVIEW"
    },
    {
        "requirement_id": "CV02",
        "requirement": "Expert segmentation masks",
        "description": "Obtain expert-reviewed pixel-level lesion masks with annotation provenance.",
        "current_status": "NOT_CONFIRMED"
    },
    {
        "requirement_id": "CV03",
        "requirement": "Independent evaluation data",
        "description": "Use a held-out dataset appropriate to the intended evaluation claim.",
        "current_status": "DEVELOPMENT_SPLIT_AVAILABLE"
    },
    {
        "requirement_id": "CV04",
        "requirement": "Lesion and patient independence",
        "description": "Prevent related lesions or patients from crossing data partitions.",
        "current_status": "LESION_DISJOINT_VERIFIED_PATIENT_ID_UNAVAILABLE"
    },
    {
        "requirement_id": "CV05",
        "requirement": "Prediction provenance",
        "description": "Preserve model version, configuration, timestamps and prediction outputs.",
        "current_status": "NOT_ESTABLISHED_FOR_CLINICAL_EVALUATION"
    },
    {
        "requirement_id": "CV06",
        "requirement": "Predefined evaluation protocol",
        "description": "Specify endpoints, inclusion criteria, exclusions and analysis methods before evaluation.",
        "current_status": "PROTOCOL_DRAFT_REQUIRED"
    },
    {
        "requirement_id": "CV07",
        "requirement": "Qualified clinical oversight",
        "description": "Require qualified clinicians to review reference standards and study findings.",
        "current_status": "MANDATORY"
    },
    {
        "requirement_id": "CV08",
        "requirement": "External validation",
        "description": "Assess generalizability on an independent, appropriately sourced dataset.",
        "current_status": "NOT_PERFORMED"
    }
]

protocol_df = pd.DataFrame(protocol_records)

# -------------------------------------------------------------------------
# 3. Define current dataset readiness
# -------------------------------------------------------------------------

readiness_records = [
    {
        "area": "Dataset file integrity",
        "status": "VERIFIED",
        "evidence": "Metadata SHA-256 hashes match supplied manifests."
    },
    {
        "area": "Image-to-metadata mapping",
        "status": "VERIFIED",
        "evidence": "All 9,704 images mapped to metadata and correct split."
    },
    {
        "area": "Lesion-level label consistency",
        "status": "VERIFIED",
        "evidence": "No inconsistent source labels within lesion groups."
    },
    {
        "area": "Cross-split lesion overlap",
        "status": "NO_OVERLAP_DETECTED",
        "evidence": "No lesion IDs shared between training and evaluation."
    },
    {
        "area": "Patient-level independence",
        "status": "NOT_ASSESSABLE",
        "evidence": "Patient identifiers were not found in the audited metadata schema."
    },
    {
        "area": "Expert segmentation references",
        "status": "UNCONFIRMED",
        "evidence": "No verified expert pixel-level masks identified."
    },
    {
        "area": "Clinical label provenance",
        "status": "REQUIRES_REVIEW",
        "evidence": "Source metadata labels are available, but independent clinical confirmation is not established."
    },
    {
        "area": "Model clinical predictions",
        "status": "NOT_ESTABLISHED",
        "evidence": "No independently validated clinical prediction dataset supplied."
    },
    {
        "area": "External clinical validation",
        "status": "NOT_PERFORMED",
        "evidence": "No external clinical validation study completed."
    }
]

readiness_df = pd.DataFrame(readiness_records)

# -------------------------------------------------------------------------
# 4. Safety and study-use constraints
# -------------------------------------------------------------------------

safety_constraints = {
    "automatic_acceptance": False,
    "independent_diagnosis": False,
    "automated_clinical_clearance": False,
    "mandatory_human_review": True,
    "qualified_clinician_final_decision": True,
    "experimental_masks_as_ground_truth": False,
    "source_dx_labels_as_independently_confirmed_diagnoses": False,
    "clinical_metrics_without_verified_references": False
}

# -------------------------------------------------------------------------
# 5. Overall readiness decision
# -------------------------------------------------------------------------

overall_status = "NOT_READY_FOR_CLINICAL_VALIDATION"

summary = {
    "phase": 12,
    "cell": 6,
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "overall_status": overall_status,
    "metric_readiness": metric_records,
    "clinical_protocol_requirements": protocol_records,
    "dataset_readiness": readiness_records,
    "safety_constraints": safety_constraints,
    "clinical_metrics_calculated": False,
    "clinical_validation_performed": False,
    "clinical_validation_status": "NOT_CLINICALLY_VALIDATED"
}

# -------------------------------------------------------------------------
# 6. Save reports
# -------------------------------------------------------------------------

metric_df.to_csv(
    REPORT_DIR / "phase12_metric_readiness.csv",
    index=False
)

protocol_df.to_csv(
    REPORT_DIR / "phase12_clinical_validation_protocol.csv",
    index=False
)

readiness_df.to_csv(
    REPORT_DIR / "phase12_clinical_dataset_readiness.csv",
    index=False
)

with open(
    OUTPUT_DIR / "phase12_clinical_validation_readiness.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(summary, f, indent=2)

# -------------------------------------------------------------------------
# 7. Display results
# -------------------------------------------------------------------------

print("\n" + "=" * 80)
print("PHASE 12 — CELL 6 RESULTS")
print("=" * 80)

print("\nCLINICAL METRIC READINESS")
display(metric_df)

print("\nVALIDATION PROTOCOL REQUIREMENTS")
display(protocol_df)

print("\nDATASET READINESS")
display(readiness_df)

print("\nOVERALL STATUS:", overall_status)
print("Clinical metrics calculated: False")
print("Clinical validation performed: False")

print("\nSAFETY CONSTRAINTS")
print(json.dumps(safety_constraints, indent=2))

print("\nNo source dataset files modified.")

print("\nSaved reports:")
print(REPORT_DIR / "phase12_metric_readiness.csv")
print(REPORT_DIR / "phase12_clinical_validation_protocol.csv")
print(REPORT_DIR / "phase12_clinical_dataset_readiness.csv")
print(OUTPUT_DIR / "phase12_clinical_validation_readiness.json")


PHASE 12 — CELL 6 RESULTS

CLINICAL METRIC READINESS


,metric,task,required_reference,evidence_available,readiness,reason
0,Dice coefficient,Lesion segmentation,Verified expert pixel-level segmentation masks,False,NOT_READY,Expert reference masks have not been confirmed.
1,Intersection over Union (IoU),Lesion segmentation,Verified expert pixel-level segmentation masks,False,NOT_READY,Expert reference masks have not been confirmed.
2,Sensitivity,Classification,Verified reference labels and held-out model p...,False,NOT_READY,No independently validated prediction set is a...
3,Specificity,Classification,Verified reference labels and held-out model p...,False,NOT_READY,No independently validated prediction set is a...
4,ROC-AUC,Classification,Verified reference labels and model probabilit...,False,NOT_READY,Validated reference labels and probability out...
5,Precision,Classification,Verified reference labels and held-out model p...,False,NOT_READY,No independently validated prediction set is a...
6,F1-score,Classification,Verified reference labels and held-out model p...,False,NOT_READY,No independently validated prediction set is a...
7,Calibration,Classification,Verified labels and representative predicted p...,False,NOT_READY,Clinical reference labels and validated probab...



VALIDATION PROTOCOL REQUIREMENTS


,requirement_id,requirement,description,current_status
0,CV01,Verified reference labels,"Document label definitions, provenance and qua...",REQUIRES_REVIEW
1,CV02,Expert segmentation masks,Obtain expert-reviewed pixel-level lesion mask...,NOT_CONFIRMED
2,CV03,Independent evaluation data,Use a held-out dataset appropriate to the inte...,DEVELOPMENT_SPLIT_AVAILABLE
3,CV04,Lesion and patient independence,Prevent related lesions or patients from cross...,LESION_DISJOINT_VERIFIED_PATIENT_ID_UNAVAILABLE
4,CV05,Prediction provenance,"Preserve model version, configuration, timesta...",NOT_ESTABLISHED_FOR_CLINICAL_EVALUATION
5,CV06,Predefined evaluation protocol,"Specify endpoints, inclusion criteria, exclusi...",PROTOCOL_DRAFT_REQUIRED
6,CV07,Qualified clinical oversight,Require qualified clinicians to review referen...,MANDATORY
7,CV08,External validation,"Assess generalizability on an independent, app...",NOT_PERFORMED



DATASET READINESS


,area,status,evidence
0,Dataset file integrity,VERIFIED,Metadata SHA-256 hashes match supplied manifests.
1,Image-to-metadata mapping,VERIFIED,"All 9,704 images mapped to metadata and correc..."
2,Lesion-level label consistency,VERIFIED,No inconsistent source labels within lesion gr...
3,Cross-split lesion overlap,NO_OVERLAP_DETECTED,No lesion IDs shared between training and eval...
4,Patient-level independence,NOT_ASSESSABLE,Patient identifiers were not found in the audi...
5,Expert segmentation references,UNCONFIRMED,No verified expert pixel-level masks identified.
6,Clinical label provenance,REQUIRES_REVIEW,"Source metadata labels are available, but inde..."
7,Model clinical predictions,NOT_ESTABLISHED,No independently validated clinical prediction...
8,External clinical validation,NOT_PERFORMED,No external clinical validation study completed.



OVERALL STATUS: NOT_READY_FOR_CLINICAL_VALIDATION
Clinical metrics calculated: False
Clinical validation performed: False

SAFETY CONSTRAINTS
{
  "automatic_acceptance": false,
  "independent_diagnosis": false,
  "automated_clinical_clearance": false,
  "mandatory_human_review": true,
  "qualified_clinician_final_decision": true,
  "experimental_masks_as_ground_truth": false,
  "source_dx_labels_as_independently_confirmed_diagnoses": false,
  "clinical_metrics_without_verified_references": false
}

No source dataset files modified.

Saved reports:
/content/drive_phase12/MyDrive/Veloura_Project/reports/clinical_validation/phase12_metric_readiness.csv
/content/drive_phase12/MyDrive/Veloura_Project/reports/clinical_validation/phase12_clinical_validation_protocol.csv
/content/drive_phase12/MyDrive/Veloura_Project/reports/clinical_validation/phase12_clinical_dataset_readiness.csv
/content/drive_phase12/MyDrive/Veloura_Project/outputs/clinical_validation/phase12_clinical_validation_readines

In [ ]:
# =============================================================================
# VELOURA — PHASE 12: CORRECTED FINAL VERIFICATION
# =============================================================================

import json
import hashlib
import pandas as pd
from pathlib import Path
from datetime import datetime, timezone

PROJECT_DIR = Path(
    "/content/drive_phase12/MyDrive/Veloura_Project"
)

OUTPUT_DIR = PROJECT_DIR / "outputs" / "clinical_validation"
REPORT_DIR = PROJECT_DIR / "reports" / "clinical_validation"

checks = []

def add_check(name, condition, detail):
    checks.append({
        "check": name,
        "passed": bool(condition),
        "detail": detail
    })

def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

# -------------------------------------------------------------------------
# 1. Required artifacts
# -------------------------------------------------------------------------

required_files = [
    "outputs/clinical_validation/phase12_dataset_inventory_summary.json",
    "outputs/clinical_validation/phase12_metadata_audit_summary.json",
    "outputs/clinical_validation/phase12_lesion_and_provenance_summary.json",
    "outputs/clinical_validation/phase12_documentation_provenance_summary.json",
    "outputs/clinical_validation/phase12_dataset_integrity_summary.json",
    "outputs/clinical_validation/phase12_clinical_validation_readiness.json",
    "reports/clinical_validation/phase12_dataset_file_inventory.csv",
    "reports/clinical_validation/phase12_metadata_dataset_summary.csv",
    "reports/clinical_validation/phase12_lesion_level_audit.csv",
    "reports/clinical_validation/phase12_image_metadata_matching.csv",
    "reports/clinical_validation/phase12_documentation_read_audit.csv",
    "reports/clinical_validation/phase12_full_manifest_read_audit.csv",
    "reports/clinical_validation/phase12_metadata_hash_verification.csv",
    "reports/clinical_validation/phase12_manifest_count_verification.csv",
    "reports/clinical_validation/phase12_metric_readiness.csv",
    "reports/clinical_validation/phase12_clinical_validation_protocol.csv",
    "reports/clinical_validation/phase12_clinical_dataset_readiness.csv"
]

inventory_records = []

for relative_path in required_files:
    path = PROJECT_DIR / relative_path

    inventory_records.append({
        "relative_path": relative_path,
        "exists": path.exists(),
        "size_bytes": path.stat().st_size if path.exists() else 0,
        "sha256": (
            hashlib.sha256(path.read_bytes()).hexdigest()
            if path.exists() else None
        )
    })

inventory_df = pd.DataFrame(inventory_records)

add_check(
    "All required Phase 12 artifacts exist",
    inventory_df["exists"].all(),
    f"{int(inventory_df['exists'].sum())}/{len(required_files)} artifacts found"
)

# -------------------------------------------------------------------------
# 2. Load phase summaries
# -------------------------------------------------------------------------

dataset_summary = load_json(
    OUTPUT_DIR / "phase12_dataset_inventory_summary.json"
)

metadata_summary = load_json(
    OUTPUT_DIR / "phase12_metadata_audit_summary.json"
)

lesion_summary = load_json(
    OUTPUT_DIR / "phase12_lesion_and_provenance_summary.json"
)

integrity_summary = load_json(
    OUTPUT_DIR / "phase12_dataset_integrity_summary.json"
)

readiness_summary = load_json(
    OUTPUT_DIR / "phase12_clinical_validation_readiness.json"
)

# -------------------------------------------------------------------------
# 3. Independently verify metadata CSV counts
# -------------------------------------------------------------------------

train_csv = (
    PROJECT_DIR /
    "datasets/training/extracted/Health/training/metadata.csv"
)

eval_csv = (
    PROJECT_DIR /
    "datasets/evaluation/extracted/Health/eval/metadata.csv"
)

train_df = pd.read_csv(train_csv)
eval_df = pd.read_csv(eval_csv)

train_count = len(train_df)
eval_count = len(eval_df)

train_unique = train_df["image_id"].nunique()
eval_unique = eval_df["image_id"].nunique()

add_check(
    "Training metadata record count",
    train_count == 7762
    and lesion_summary.get("training_metadata_records") == 7762,
    f"CSV={train_count}; Cell 3 summary={lesion_summary.get('training_metadata_records')}"
)

add_check(
    "Evaluation metadata record count",
    eval_count == 1942
    and lesion_summary.get("evaluation_metadata_records") == 1942,
    f"CSV={eval_count}; Cell 3 summary={lesion_summary.get('evaluation_metadata_records')}"
)

add_check(
    "Training image IDs unique",
    train_unique == train_count,
    f"{train_unique}/{train_count} unique image IDs"
)

add_check(
    "Evaluation image IDs unique",
    eval_unique == eval_count,
    f"{eval_unique}/{eval_count} unique image IDs"
)

# -------------------------------------------------------------------------
# 4. Dataset integrity
# -------------------------------------------------------------------------

add_check(
    "Metadata hashes match manifests",
    integrity_summary.get("all_metadata_hashes_match") is True,
    "Both metadata SHA-256 hashes match supplied manifest values"
)

add_check(
    "Manifest count checks passed",
    integrity_summary.get("all_count_checks_pass") is True,
    "Metadata and physical image counts match manifest"
)

add_check(
    "Training and evaluation metadata are distinct",
    integrity_summary.get("metadata_files_distinct") is True,
    "Metadata hashes are different"
)

# -------------------------------------------------------------------------
# 5. Lesion-level consistency and mapping
# -------------------------------------------------------------------------

add_check(
    "Training lesion-label consistency",
    lesion_summary.get("training_inconsistent_lesions") == 0,
    f"Inconsistent groups: {lesion_summary.get('training_inconsistent_lesions')}"
)

add_check(
    "Evaluation lesion-label consistency",
    lesion_summary.get("evaluation_inconsistent_lesions") == 0,
    f"Inconsistent groups: {lesion_summary.get('evaluation_inconsistent_lesions')}"
)

add_check(
    "Training image mapping",
    lesion_summary.get("training_images_correctly_mapped") == 7762,
    f"Mapped: {lesion_summary.get('training_images_correctly_mapped')}/7762"
)

add_check(
    "Evaluation image mapping",
    lesion_summary.get("evaluation_images_correctly_mapped") == 1942,
    f"Mapped: {lesion_summary.get('evaluation_images_correctly_mapped')}/1942"
)

add_check(
    "Six reference cases remain training-only",
    metadata_summary.get("reference_cases") == 6
    and metadata_summary.get("reference_cases_with_metadata") == 6,
    "All six prior reference cases have training metadata"
)

# -------------------------------------------------------------------------
# 6. Clinical metric readiness
# -------------------------------------------------------------------------

metric_df = pd.read_csv(
    REPORT_DIR / "phase12_metric_readiness.csv"
)

metrics_blocked = (
    len(metric_df) == 8
    and (metric_df["readiness"] == "NOT_READY").all()
)

add_check(
    "Clinical metrics correctly blocked",
    metrics_blocked,
    "All eight metrics remain NOT_READY"
)

# -------------------------------------------------------------------------
# 7. Safety invariants
# -------------------------------------------------------------------------

safety = readiness_summary.get("safety_constraints", {})

required_safety = {
    "automatic_acceptance": False,
    "independent_diagnosis": False,
    "automated_clinical_clearance": False,
    "mandatory_human_review": True,
    "qualified_clinician_final_decision": True,
    "experimental_masks_as_ground_truth": False,
    "source_dx_labels_as_independently_confirmed_diagnoses": False,
    "clinical_metrics_without_verified_references": False
}

safety_pass = all(
    safety.get(key) == expected
    for key, expected in required_safety.items()
)

add_check(
    "All clinical safety constraints preserved",
    safety_pass,
    "All eight safety invariants match required values"
)

add_check(
    "Clinical validation not falsely claimed",
    readiness_summary.get("clinical_validation_performed") is False
    and readiness_summary.get("overall_status")
        == "NOT_READY_FOR_CLINICAL_VALIDATION",
    "Clinical validation remains NOT PERFORMED"
)

# -------------------------------------------------------------------------
# 8. Final verification status
# -------------------------------------------------------------------------

verification_df = pd.DataFrame(checks)

passed = int(verification_df["passed"].sum())
failed = int((~verification_df["passed"]).sum())

phase12_status = (
    "PHASE_12_TECHNICALLY_VERIFIED"
    if failed == 0
    else "PHASE_12_VERIFICATION_REQUIRES_REVIEW"
)

verification_df.to_csv(
    REPORT_DIR / "phase12_final_verification_checks.csv",
    index=False
)

inventory_df.to_csv(
    REPORT_DIR / "phase12_final_artifact_inventory.csv",
    index=False
)

completion_summary = {
    "phase": 12,
    "phase_name": "Clinical Validation and Dataset Readiness",
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "phase_status": phase12_status,
    "technical_checks_passed": passed,
    "technical_checks_failed": failed,
    "required_artifacts_found": int(inventory_df["exists"].sum()),
    "required_artifacts_expected": len(required_files),
    "training_metadata_records": train_count,
    "evaluation_metadata_records": eval_count,
    "training_unique_image_ids": train_unique,
    "evaluation_unique_image_ids": eval_unique,
    "dataset_integrity": "VERIFIED",
    "lesion_level_consistency": "VERIFIED",
    "image_metadata_mapping": "VERIFIED",
    "clinical_metric_readiness": "NOT_READY",
    "expert_segmentation_annotations": "UNCONFIRMED",
    "clinical_label_provenance": "REQUIRES_REVIEW",
    "clinical_validation": "NOT_CLINICALLY_VALIDATED",
    "external_validation": "NOT_PERFORMED",
    "automatic_acceptance": False,
    "independent_diagnosis": False,
    "automated_clinical_clearance": False,
    "mandatory_human_review": True,
    "qualified_clinician_final_decision": True
}

with open(
    OUTPUT_DIR / "phase12_completion_report.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(completion_summary, f, indent=2)

pd.DataFrame([completion_summary]).to_csv(
    REPORT_DIR / "phase12_completion_summary.csv",
    index=False
)

# -------------------------------------------------------------------------
# 9. Display results
# -------------------------------------------------------------------------

print("\n" + "=" * 80)
print("VELOURA — PHASE 12 CORRECTED FINAL VERIFICATION")
print("=" * 80)

display(verification_df)

print("\nARTIFACT INVENTORY")
display(inventory_df)

print("\n" + "=" * 80)
print("PHASE 12 STATUS:", phase12_status)
print("=" * 80)

print(f"Checks passed: {passed}")
print(f"Checks failed: {failed}")
print(
    f"Artifacts found: {int(inventory_df['exists'].sum())}"
    f"/{len(required_files)}"
)

print("\nTraining records:", train_count)
print("Evaluation records:", eval_count)
print("Clinical validation: NOT CLINICALLY VALIDATED")
print("Expert masks: UNCONFIRMED")
print("Clinical metrics: NOT READY")
print("Automatic acceptance: DISABLED")
print("Independent diagnosis: DISABLED")
print("Mandatory human review: ENABLED")
print("Qualified clinician: FINAL DECISION OWNER")

print("\nFinal reports saved successfully.")


VELOURA — PHASE 12 CORRECTED FINAL VERIFICATION


,check,passed,detail
0,All required Phase 12 artifacts exist,True,17/17 artifacts found
1,Training metadata record count,True,CSV=7762; Cell 3 summary=7762
2,Evaluation metadata record count,True,CSV=1942; Cell 3 summary=1942
3,Training image IDs unique,True,7762/7762 unique image IDs
4,Evaluation image IDs unique,True,1942/1942 unique image IDs
5,Metadata hashes match manifests,True,Both metadata SHA-256 hashes match supplied ma...
6,Manifest count checks passed,True,Metadata and physical image counts match manifest
7,Training and evaluation metadata are distinct,True,Metadata hashes are different
8,Training lesion-label consistency,True,Inconsistent groups: 0
9,Evaluation lesion-label consistency,True,Inconsistent groups: 0



ARTIFACT INVENTORY


,relative_path,exists,size_bytes,sha256
0,outputs/clinical_validation/phase12_dataset_in...,True,574,22869674c75b77dcb80564b7bfbdd7d181461e84dbc22a...
1,outputs/clinical_validation/phase12_metadata_a...,True,1289,48f8d815d3ee05fb0b46095b2764993d405fc0d55e2902...
2,outputs/clinical_validation/phase12_lesion_and...,True,625,d2bcbf165855b5c6073386366893017aef075950c9206c...
3,outputs/clinical_validation/phase12_documentat...,True,1574,c7e007d0aed3604a87705dbf50477fda663846a5bef743...
4,outputs/clinical_validation/phase12_dataset_in...,True,2015,b3f9689af04eb2a15a15bff45e953499205606ee4a9f10...
5,outputs/clinical_validation/phase12_clinical_v...,True,6741,18b408da3776ffdfe5f0d1f90a7b66d7e051bebb91ebe7...
6,reports/clinical_validation/phase12_dataset_fi...,True,937995,5209c15e5567066149e5585f366f68867d51ca33f9d412...
7,reports/clinical_validation/phase12_metadata_d...,True,253,5e110bfdd9d6b875693906fdd7ff0a6fb10f034f2978a2...
8,reports/clinical_validation/phase12_lesion_lev...,True,371577,0886d59ce4ed18834b05705d65ca5a8e50f1b6f02b3ad3...
9,reports/clinical_validation/phase12_image_meta...,True,1009344,f14f1c62ff604d305304512dffd5ea1d9e6c36b0910941...



PHASE 12 STATUS: PHASE_12_TECHNICALLY_VERIFIED
Checks passed: 16
Checks failed: 0
Artifacts found: 17/17

Training records: 7762
Evaluation records: 1942
Clinical validation: NOT CLINICALLY VALIDATED
Expert masks: UNCONFIRMED
Clinical metrics: NOT READY
Automatic acceptance: DISABLED
Independent diagnosis: DISABLED
Mandatory human review: ENABLED
Qualified clinician: FINAL DECISION OWNER

Final reports saved successfully.


In [ ]:
# =============================================================================
# PHASE 12 — FINAL VERIFICATION DIAGNOSTIC
# Inspect actual JSON keys before correcting verification logic
# =============================================================================

import json
from pathlib import Path

PROJECT_DIR = Path(
    "/content/drive_phase12/MyDrive/Veloura_Project"
)

OUTPUT_DIR = PROJECT_DIR / "outputs" / "clinical_validation"

files_to_inspect = [
    "phase12_metadata_audit_summary.json",
    "phase12_dataset_inventory_summary.json",
    "phase12_lesion_and_provenance_summary.json",
]

for filename in files_to_inspect:
    path = OUTPUT_DIR / filename

    print("\n" + "=" * 80)
    print(filename)
    print("=" * 80)

    if not path.exists():
        print("FILE NOT FOUND")
        continue

    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)

    print("Top-level type:", type(data).__name__)

    if isinstance(data, dict):
        print("\nAvailable keys:")
        for key, value in data.items():
            if isinstance(value, (dict, list)):
                preview = (
                    f"{type(value).__name__}, "
                    f"length={len(value)}"
                )
            else:
                preview = repr(value)

            print(f"{key}: {preview}")

    else:
        print(json.dumps(data, indent=2, default=str)[:5000])

print("\nDiagnostic completed. No files modified.")


phase12_metadata_audit_summary.json
Top-level type: dict

Available keys:
project: 'Veloura Agentic AI 2026'
phase: 12
cell: 2
timestamp_utc: '2026-10-30T21:37:45.782119+00:00'
metadata_files_loaded: 2
dataset_summaries: list, length=2
reference_cases: 6
reference_cases_with_metadata: 6
train_evaluation_overlap: list, length=2
expert_annotation_status: 'NOT_CONFIRMED'
clinical_validation_status: 'NOT_CLINICALLY_VALIDATED'
source_metadata_modified: False

phase12_dataset_inventory_summary.json
Top-level type: dict

Available keys:
project: 'Veloura Agentic AI 2026'
phase: 12
cell: 1
timestamp_utc: '2026-10-30T21:36:49.617397+00:00'
dataset_directory: '/content/drive_phase12/MyDrive/Veloura_Project/datasets'
total_dataset_files: 9712
image_file_count: 9704
metadata_file_count: 4
annotation_candidate_count: 0
experimental_mask_file_count: 24
reference_case_count: 6
reference_images_found: 6
expert_annotation_status: 'NOT_YET_CONFIRMED'
clinical_validation_status: 'NOT_CLINICALLY_VALIDATE